# Valkey as a job queue and admission gate for llm-d

This notebook points [llm-d](https://llm-d.ai)'s async batch processor at Valkey
and watches the work live in Valkey: a **sorted set** holding queued inference
jobs in priority order, and a second sorted set enforcing a per-user admission
quota. llm-d is the real-world example; the reusable pattern is one Valkey sorted
set doing two separate jobs: a priority work queue and a sliding-window rate
limiter.

llm-d is pointed at Valkey by a connection URL alone.

## The feature

A sorted set (ZSET) orders members by a score. Here one structure does two jobs:

- **Priority job queue** (`request-sortedset`): each job is a member of the
  sorted set, and its score is derived from the deadline. `ZRANGE` returns
  members in score order, so jobs come off in priority order; jobs with the
  same priority come off in the order they were submitted.
- **Admission quota** (`quota:userid:<id>`): the gate records each admitted
  request as a scored member in a per-user sorted set and counts members inside a
  time window; past the limit, requests are refused until the window rolls
  forward.

You will submit jobs, watch them queue and drain, see results come back, then
send a burst past the quota and watch the gate refuse the overflow.

## Before you start

Build the producer image and bring the stack up from the cookbook directory. The
commands below use Docker; with Podman, run the same commands with `podman` and
`podman compose` (the notebook itself picks whichever runtime is installed):

```bash
docker build -t llm-d-valkey-producer ./producer
docker compose up -d --wait
```

> ⚠️ **Security:** These examples use no authentication or TLS for simplicity.
> For any non-localhost deployment, enable authentication and TLS.
> See the [Valkey security documentation](https://valkey.io/topics/security/).

`docker compose ps` should show `valkey`, `llm-sim`, and `async-processor` up,
with `valkey` healthy. All ports are bound to `127.0.0.1`, so nothing is exposed
beyond your machine. Tear down with `docker compose down` when finished.

In [ ]:
import json
import shutil
import subprocess
import time

import requests
from glide_sync import GlideClient, GlideClientConfiguration, NodeAddress

# Valkey is published on localhost by docker-compose.yml.
VALKEY_HOST, VALKEY_PORT = "127.0.0.1", 6379
# The async processor's Prometheus metrics endpoint (also localhost).
METRICS_URL = "http://127.0.0.1:9090/metrics"
# Names the processor's transport-config.json gives the queue and result list.
QUEUE = "request-sortedset"
RESULTS = "result-list"
# The compose-created network the one-shot producer container joins. Both Docker
# Compose and Podman Compose name it "<project-dir>_default".
COMPOSE_NETWORK = "queue-and-quota_default"


def detect_engine():
    """First of docker/podman that is installed and answers `info`.

    Honors $CONTAINER_ENGINE if set. Mirrors the kv-caching cookbook so the
    notebook runs the same under Docker, Podman, or a compatible runtime."""
    import os

    forced = os.environ.get("CONTAINER_ENGINE")
    candidates = [forced] if forced else ["docker", "podman"]
    for cand in candidates:
        if cand and shutil.which(cand):
            try:
                subprocess.run([cand, "info"], check=True, capture_output=True, timeout=20)
                return cand
            except Exception:
                continue
    raise RuntimeError("no working container runtime found (tried docker, podman)")


ENGINE = detect_engine()
print("using container runtime:", ENGINE)


def connect():
    return GlideClient.create(
        GlideClientConfiguration(
            addresses=[NodeAddress(VALKEY_HOST, VALKEY_PORT)],
            request_timeout=2000,
        )
    )


valkey = connect()
assert valkey.ping() in (b"PONG", "PONG")
print("connected to Valkey at", f"{VALKEY_HOST}:{VALKEY_PORT}")

## A sorted set is already a priority queue and a rate limiter

Before involving llm-d, here is the whole idea in a few commands against Valkey
directly, no framework, just the sorted set.

A **priority queue**: add jobs with a score, pop the lowest score first.

In [ ]:
demo_q = "demo:priority-queue"
valkey.custom_command(["DEL", demo_q])

# Lower score = higher priority. Two "urgent" jobs (score 1) and a "bulk" job (5).
valkey.custom_command(["ZADD", demo_q, "5", "bulk-report"])
valkey.custom_command(["ZADD", demo_q, "1", "urgent-a"])
valkey.custom_command(["ZADD", demo_q, "1", "urgent-b"])

# Pop in priority order: ZRANGE returns lowest-score first; urgent before bulk.
order = valkey.custom_command(["ZRANGE", demo_q, "0", "-1"])
print("drain order:", [m.decode() if isinstance(m, bytes) else m for m in order])
valkey.custom_command(["DEL", demo_q])

A **sliding-window rate limiter**: record each admission as a timestamped
member, drop members older than the window, and the member count is the rate.

In [ ]:
demo_rl = "demo:rate-limit"
valkey.custom_command(["DEL", demo_rl])
LIMIT, WINDOW_S = 3, 60

def allow(now):
    # Drop admissions older than the window, then count what remains.
    valkey.custom_command(["ZREMRANGEBYSCORE", demo_rl, "-inf", f"({now - WINDOW_S}"])
    count = int(valkey.custom_command(["ZCARD", demo_rl]))
    if count >= LIMIT:
        return False
    valkey.custom_command(["ZADD", demo_rl, str(now), f"req-{now}-{count}"])
    return True

now = int(time.time())
print("admit decisions for 5 requests (limit 3):",
      [allow(now) for _ in range(5)])
valkey.custom_command(["DEL", demo_rl])

That is exactly what llm-d does with Valkey, at production shape and with a
Lua script for atomicity. The rest of the notebook watches the real thing.

## Submit jobs and watch them queue

The producer submits jobs through llm-d's public producer library, so what lands
in Valkey is exactly what the async processor expects. To *see* the queue hold
jobs, pause the processor first, submit, then look at the sorted set.

In [ ]:
def submit(n, userid=None):
    # Flags only: a leading positional arg would make Go's flag parser skip them.
    args = [
        ENGINE, "run", "--rm", "--network", COMPOSE_NETWORK,
        "llm-d-valkey-producer",
        "--redis-url", "redis://valkey:6379",
        "--request-queue", QUEUE,
        "--n", str(n), "--read-results=false",
    ]
    if userid:
        args += ["--userid", userid]
    subprocess.run(args, check=True, capture_output=True, timeout=60)


# Pause the processor so submissions accumulate instead of draining immediately.
subprocess.run([ENGINE, "compose", "pause", "async-processor"],
               check=True, capture_output=True)
valkey.custom_command(["DEL", QUEUE])
submit(4)
print("queued jobs:", int(valkey.custom_command(["ZCARD", QUEUE])))

Each member is the request as JSON. `ZRANGE ... WITHSCORES` shows the
priority ordering: the score encodes the deadline plus a monotonic enqueue
sequence, so jobs drain by priority, and ties break in submission order.

In [ ]:
rows = valkey.custom_command(["ZRANGE", QUEUE, "0", "-1", "WITHSCORES"])
# GLIDE returns ZRANGE WITHSCORES as a dict {member: score}. Sort by score to
# see the drain order (lowest score first).
for raw, score in sorted(rows.items(), key=lambda kv: kv[1]):
    job = json.loads(raw.decode() if isinstance(raw, bytes) else raw)
    print(f"seq={job['internal']['enqueue_seq']}  id={job['data']['id']}  score={score}")

## Let them drain, and read the results

Unpause the processor. It pops each job off the sorted set, forwards it to the
inference backend, and writes the result to `result-list`. The queue drains to
empty.

In [ ]:
subprocess.run([ENGINE, "compose", "unpause", "async-processor"],
               check=True, capture_output=True)

deadline = time.monotonic() + 15
while time.monotonic() < deadline and int(valkey.custom_command(["ZCARD", QUEUE])) > 0:
    time.sleep(0.5)
print("queue depth after draining:", int(valkey.custom_command(["ZCARD", QUEUE])))

# Results are pushed to a list; read them back (OpenAI-shaped bodies from the sim).
n_results = int(valkey.custom_command(["LLEN", RESULTS]))
print("results waiting:", n_results)
if n_results:
    raw = valkey.custom_command(["LINDEX", RESULTS, "0"])
    result = json.loads(raw.decode() if isinstance(raw, bytes) else raw)
    print("one result keys:", sorted(result.keys()))

## The admission quota in action

The queue's `redis-quota` gate (see `transport-config.json`) is configured
`attribute: userid`, `limit: 3`, `window: 1m`. It keys on each job's
`metadata.userid`. A job with no `userid` is always admitted, so the next cell
sets it on every job: the `submit(..., userid="alice")` call passes `userid`
through to the job's metadata, which is what the gate counts against the limit.

Submit 6 jobs as the same user against a limit of 3 and let the processor gate
them.

In [ ]:
valkey.custom_command(["FLUSHALL"])
submit(6, userid="alice")
time.sleep(7)  # let the processor evaluate the burst

# The gate's own state lives in Valkey as a sorted set: one scored member per
# admission inside the window. Its cardinality is the admitted count = the limit.
qkey = "quota:userid:alice"
qtype = valkey.custom_command(["TYPE", qkey])
print("quota key type:", qtype.decode() if isinstance(qtype, bytes) else qtype)
print("admitted in window (ZCARD):", int(valkey.custom_command(["ZCARD", qkey])))
print("window TTL remaining (s):", int(valkey.custom_command(["TTL", qkey])))

The refusals are also counted in the processor's metrics. The
`quota_exhausted` counter is non-zero: the gate refused the jobs over the limit.
(In blocking mode the processor requeues a refused job and re-evaluates it each
poll, so the counter climbs past the raw refusal count. That is the gate holding
the overflow back, working as intended.)

In [ ]:
metrics = requests.get(METRICS_URL, timeout=5).text
for line in metrics.splitlines():
    if line.startswith("llm_d_async_async_gate_decisions_total") and "quota_exhausted" in line:
        print(line)
    if line.startswith("llm_d_async_async_dispatched_requests_total"):
        print(line)

## What made this work with Valkey

Only one thing points llm-d at Valkey: the `url` in `transport-config.json`
(`redis://valkey:6379`). llm-d speaks the Redis serialization protocol, which
Valkey implements, so the async processor, the queue, and the quota gate all run
against Valkey with no code change. llm-d's own Helm chart states it is
"compatible with any Redis-protocol server including Valkey."

You saw a single Valkey primitive, the sorted set, serve as both the priority
job queue and the sliding-window admission quota behind a real distributed
inference system.

In [ ]:
valkey.close()
print(f"done — `{ENGINE} compose down` to stop the stack")